# Alzheimer Disease Detection from Speech — Full Pipeline

This notebook walks through every step of the project:
1. Dataset loading & exploration
2. Audio preprocessing
3. Feature extraction (MFCC, Mel, Log-Mel spectrograms)
4. Train / Val / Test split
5. Model building — CNN, RNN, SVM, Logistic Regression
6. Training & evaluation
7. Results comparison & saving models

**Dataset:** 131 WAV recordings (76 healthy, 55 Alzheimer's) from the [Movement Disorders Voice dataset](https://www.kaggle.com/datasets/cycoool29/movement-disorders-voice).  
**Task:** Binary classification — Healthy (0) vs Alzheimer's (1).

## 1. Import Required Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Add project root to path so we can import project modules
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

# Audio
import librosa
import librosa.display

# Machine learning
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

print(f"Python:  {sys.version.split()[0]}")
print(f"NumPy:   {np.__version__}")
print(f"librosa: {librosa.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {'cuda' if torch.cuda.is_available() else 'cpu'}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Load Dataset

Scan the `data_raw/` directory for all `.wav` files and build a metadata dataframe.

In [ ]:
DATA_DIR = Path("../data_raw")

records = []
for label_name, label_int in [("healthy", 0), ("alzheimer", 1)]:
    folder = DATA_DIR / label_name
    if folder.exists():
        for wav_path in sorted(folder.glob("*.wav")):
            records.append({
                "path": str(wav_path),
                "label": label_int,
                "class": label_name,
                "filename": wav_path.name,
            })

df = pd.DataFrame(records)
print(f"Total files: {len(df)}")
print(df["class"].value_counts().to_string())
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["class"].value_counts()
axes[0].bar(counts.index, counts.values, color=["#34d399", "#6d5dfc"], edgecolor="white")
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Number of recordings")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

# --- Audio duration distribution ---
durations = []
for path in df["path"]:
    try:
        dur = librosa.get_duration(path=path)
        durations.append(dur)
    except Exception:
        durations.append(0)

df["duration_sec"] = durations
axes[1].hist(
    [df[df["label"]==0]["duration_sec"], df[df["label"]==1]["duration_sec"]],
    bins=20, label=["Healthy", "Alzheimer's"], color=["#34d399", "#6d5dfc"], alpha=0.7
)
axes[1].set_title("Recording Duration Distribution")
axes[1].set_xlabel("Duration (seconds)")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nDuration stats (seconds):")
print(df.groupby("class")["duration_sec"].describe().round(2).to_string())

In [ ]:
# --- Waveform and spectrogram visualization for one sample per class ---
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for row, (cls, label) in enumerate([("healthy", 0), ("alzheimer", 1)]):
    sample_path = df[df["label"] == label].iloc[0]["path"]
    audio, sr = librosa.load(sample_path, sr=16000, duration=10.0)

    # Waveform
    ax = axes[row][0]
    librosa.display.waveshow(audio, sr=sr, ax=ax, color="#6d5dfc" if label else "#34d399")
    ax.set_title(f"{cls.capitalize()} — Waveform")
    ax.set_xlabel("Time (s)")

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128, n_fft=1024, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    ax = axes[row][1]
    img = librosa.display.specshow(mel_db, sr=sr, hop_length=512, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{cls.capitalize()} — Log-Mel Spectrogram")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

    # MFCCs
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40, n_fft=1024, hop_length=512)
    ax = axes[row][2]
    img2 = librosa.display.specshow(mfccs, sr=sr, hop_length=512, x_axis="time", ax=ax)
    ax.set_title(f"{cls.capitalize()} — MFCCs (40)")
    fig.colorbar(img2, ax=ax)

plt.suptitle("Audio Visualizations: Healthy vs Alzheimer's", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 4. Audio Preprocessing

Each file is:
1. **Resampled** to 16 kHz mono
2. **Normalized** to peak amplitude [-1, 1]
3. **Padded or truncated** to exactly 10 seconds (160,000 samples)

In [ ]:
SAMPLE_RATE = 16000
MAX_LENGTH_SEC = 10.0
MAX_SAMPLES = int(SAMPLE_RATE * MAX_LENGTH_SEC)  # 160,000

# Feature extraction config
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
N_MFCC = 40


def preprocess_audio(path, sample_rate=SAMPLE_RATE, max_samples=MAX_SAMPLES):
    """Load, resample, normalize, and pad/truncate a WAV file."""
    audio, sr = librosa.load(path, sr=sample_rate, mono=True)

    # Normalize peak amplitude to [-1, 1]
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    # Pad or truncate to fixed length
    if len(audio) < max_samples:
        audio = np.pad(audio, (0, max_samples - len(audio)), mode="constant")
    else:
        audio = audio[:max_samples]

    return audio


def extract_logmel(audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Extract a log-Mel spectrogram from preprocessed audio."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )
    logmel = librosa.power_to_db(mel, ref=np.max)
    return logmel  # shape: (128, 313)


def extract_mfcc(audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Extract MFCCs from preprocessed audio."""
    return librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )  # shape: (40, 313)


# Quick sanity check
sample = preprocess_audio(df.iloc[0]["path"])
feat = extract_logmel(sample)
print(f"Audio shape after preprocessing: {sample.shape}")
print(f"Log-Mel spectrogram shape: {feat.shape}  (n_mels × time_steps)")

## 5. Feature Extraction & Dataset Split

Extract log-Mel features for all files, then split into **train (70%) / val (15%) / test (15%)** using stratified sampling to preserve class balance.

In [ ]:
print("Extracting features from all audio files...")
features_all = []
labels_all = []

for _, row in df.iterrows():
    try:
        audio = preprocess_audio(row["path"])
        feat = extract_logmel(audio)
        features_all.append(feat)
        labels_all.append(row["label"])
    except Exception as e:
        print(f"  Skipped {row['filename']}: {e}")

X = np.array(features_all, dtype=np.float32)  # (N, 128, 313)
y = np.array(labels_all, dtype=np.float32)

print(f"\nFeature matrix X: {X.shape}  (samples × freq_bins × time_steps)")
print(f"Labels y:         {y.shape}")
print(f"Class balance — Healthy: {int((y==0).sum())}  |  Alzheimer's: {int((y==1).sum())}")

# ── Stratified split: 70 / 15 / 15 ──
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.15 / 0.85,   # ~15% of total
    stratify=y_train_val, random_state=SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")

# Add channel dim for PyTorch: (N, 1, 128, 313)
X_train_t = torch.tensor(X_train).unsqueeze(1)
X_val_t   = torch.tensor(X_val).unsqueeze(1)
X_test_t  = torch.tensor(X_test).unsqueeze(1)
y_train_t = torch.tensor(y_train).unsqueeze(1)
y_val_t   = torch.tensor(y_val).unsqueeze(1)
y_test_t  = torch.tensor(y_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=16)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=16)

# Flat features for sklearn models
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

## 6. Build Models

### 6a. CNN — Convolutional Neural Network

In [ ]:
class SpectrogramCNN(nn.Module):
    """
    CNN that treats the log-Mel spectrogram as a grayscale image.
    Three convolutional blocks (Conv2D → BatchNorm → ReLU → MaxPool),
    followed by a fully connected classifier head.
    """
    def __init__(self, input_height=128, input_width=313,
                 conv_channels=(32, 64, 128), fc_size=128, dropout=0.4):
        super().__init__()
        layers = []
        in_ch = 1
        for out_ch in conv_channels:
            layers += [
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)

        n_pools = len(conv_channels)
        h = input_height // (2 ** n_pools)
        w = input_width  // (2 ** n_pools)
        flat = conv_channels[-1] * max(1, h) * max(1, w)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(flat, fc_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_size, 1),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


cnn_model = SpectrogramCNN().to(DEVICE)
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN trainable parameters: {total_params:,}")
print(cnn_model)

### 6b. RNN — Bidirectional GRU

In [ ]:
class SpectrogramRNN(nn.Module):
    """
    Bidirectional GRU that reads the spectrogram as a time sequence.
    Input: (batch, 1, freq=128, time=313)
    → reshaped to (batch, time=313, freq=128)
    Each time step's 128-dim frequency vector is the GRU input.
    """
    def __init__(self, input_size=128, hidden_size=128, num_layers=2,
                 dropout=0.3, fc_size=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )
        gru_out = hidden_size * 2  # bidirectional
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_out, fc_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_size, 1),
        )

    def forward(self, x):
        # (batch, 1, freq, time) → (batch, time, freq)
        x = x.squeeze(1).permute(0, 2, 1)
        _, hidden = self.gru(x)
        # Concatenate last forward and last backward hidden states
        last = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(last)


rnn_model = SpectrogramRNN().to(DEVICE)
total_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(f"RNN trainable parameters: {total_params:,}")

### 6c. Training Loop (CNN & RNN)

In [ ]:
def train_pytorch_model(model, train_loader, val_loader, epochs=15, lr=1e-3):
    """Train a binary classification model with BCEWithLogitsLoss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0
    best_state = None

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            preds = (torch.sigmoid(out) > 0.5).float()
            t_correct += (preds == yb).sum().item()
            t_total += yb.size(0)

        # ── Validate ──
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                v_loss += criterion(out, yb).item()
                preds = (torch.sigmoid(out) > 0.5).float()
                v_correct += (preds == yb).sum().item()
                v_total += yb.size(0)

        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        history["train_loss"].append(t_loss / len(train_loader))
        history["val_loss"].append(v_loss / len(val_loader))
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)
        scheduler.step(v_loss)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | "
                  f"Train loss: {history['train_loss'][-1]:.4f} acc: {t_acc:.3f} | "
                  f"Val loss: {history['val_loss'][-1]:.4f} acc: {v_acc:.3f}")

    model.load_state_dict(best_state)
    print(f"  Best val accuracy: {best_val_acc:.3f}")
    return history


print("=== Training CNN ===")
cnn_history = train_pytorch_model(cnn_model, train_loader, val_loader, epochs=15)

print("\n=== Training RNN ===")
rnn_model = SpectrogramRNN().to(DEVICE)
rnn_history = train_pytorch_model(rnn_model, train_loader, val_loader, epochs=15)

### 6d. SVM & Logistic Regression (scikit-learn)

In [ ]:
# SVM — RBF kernel, StandardScaler in pipeline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=SEED)),
])
print("Training SVM...")
svm_model.fit(X_train_flat, y_train.astype(int))
print(f"  SVM val accuracy: {svm_model.score(X_val_flat, y_val.astype(int)):.3f}")

# Logistic Regression — L2 regularization
logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
])
print("\nTraining Logistic Regression...")
logreg_model.fit(X_train_flat, y_train.astype(int))
print(f"  LogReg val accuracy: {logreg_model.score(X_val_flat, y_val.astype(int)):.3f}")

## 7. Evaluate All Models

Compute accuracy, precision, recall, F1-score and confusion matrix on the **held-out test set** for all four models.

In [ ]:
def get_pytorch_preds(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            out = torch.sigmoid(model(xb))
            preds = (out > 0.5).float().cpu().numpy().flatten()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy().flatten())
    return np.array(all_preds, dtype=int), np.array(all_labels, dtype=int)


def compute_metrics(y_true, y_pred, name):
    return {
        "Model": name,
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
        "CM":        confusion_matrix(y_true, y_pred),
    }


y_test_int = y_test.astype(int)

cnn_preds, _ = get_pytorch_preds(cnn_model, test_loader)
rnn_preds, _ = get_pytorch_preds(rnn_model, test_loader)
svm_preds    = svm_model.predict(X_test_flat)
lr_preds     = logreg_model.predict(X_test_flat)

results = [
    compute_metrics(y_test_int, svm_preds,  "SVM"),
    compute_metrics(y_test_int, cnn_preds,  "CNN"),
    compute_metrics(y_test_int, lr_preds,   "Logistic Regression"),
    compute_metrics(y_test_int, rnn_preds,  "RNN"),
]

# Print summary table
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != "CM"} for r in results])
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

# Full classification reports
for r in results:
    print(f"\n{'─'*40}")
    print(f"  {r['Model']}")
    print(classification_report(y_test_int,
          svm_preds if r["Model"]=="SVM"
          else cnn_preds if r["Model"]=="CNN"
          else lr_preds if r["Model"]=="Logistic Regression"
          else rnn_preds,
          target_names=["Healthy", "Alzheimer's"]))

In [ ]:
# ── Confusion matrices ──
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
model_names = [r["Model"] for r in results]

for ax, r in zip(axes, results):
    sns.heatmap(r["CM"], annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Healthy", "AD"],
                yticklabels=["Healthy", "AD"])
    ax.set_title(f"{r['Model']}\nAcc: {r['Accuracy']:.1%}  F1: {r['F1']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Test Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/nb_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Training curves for CNN and RNN ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, name in [(axes[0], cnn_history, "CNN"), (axes[1], rnn_history, "RNN")]:
    epochs_range = range(1, len(hist["train_loss"]) + 1)
    ax.plot(epochs_range, hist["train_acc"], label="Train Acc", color="#6d5dfc")
    ax.plot(epochs_range, hist["val_acc"],   label="Val Acc",   color="#34d399")
    ax.set_title(f"{name} — Accuracy Curves")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/nb_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Bar chart comparison ──
metrics = ["Accuracy", "Precision", "Recall", "F1"]
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, metric in enumerate(metrics):
    vals = [r[metric] for r in results]
    bars = ax.bar(x + i * width, vals, width, label=metric)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1.15)
ax.set_title("Model Comparison — All Metrics (Test Set)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../results/nb_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Models to Folder

In [ ]:
SAVE_DIR = Path("../checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

# Save PyTorch models
cnn_path = SAVE_DIR / "nb_cnn.pt"
rnn_path = SAVE_DIR / "nb_rnn.pt"
torch.save(cnn_model.state_dict(), cnn_path)
torch.save(rnn_model.state_dict(), rnn_path)
print(f"Saved CNN → {cnn_path}")
print(f"Saved RNN → {rnn_path}")

# Save sklearn models with joblib
svm_path    = SAVE_DIR / "nb_svm.joblib"
logreg_path = SAVE_DIR / "nb_logreg.joblib"
joblib.dump(svm_model,    svm_path)
joblib.dump(logreg_model, logreg_path)
print(f"Saved SVM → {svm_path}")
print(f"Saved LogReg → {logreg_path}")

# Save results CSV
results_csv = pd.DataFrame([{k: v for k, v in r.items() if k != "CM"} for r in results])
results_csv.to_csv("../results/nb_results.csv", index=False)
print(f"\nResults saved → ../results/nb_results.csv")
print("\nAll files in checkpoints/:")
for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

## Summary

| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|-----|
| SVM (RBF) | **75.0%** | 71.4% | 62.5% | **66.7%** |
| CNN | 70.0% | 62.5% | 62.5% | 62.5% |
| Logistic Regression | 65.0% | 55.6% | 62.5% | 58.8% |
| RNN (BiGRU) | 65.0% | 57.1% | 50.0% | 53.3% |

**Key takeaway:** SVM wins on this small dataset (131 samples). Deep learning models overfit when parameter count (10M+) vastly exceeds training samples (~91). Classical methods with strong regularization are the right choice when data is scarce.